In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [3]:
from src.data_loader import load_data

(
    X_train,
    X_test,
    y_train,
    y_test,
    FEATURE_COLS
) = load_data()


Total Features: 39
Train Rows : 5245
Test Rows  : 2485
Scaler saved


In [5]:
import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

def objective_rf(trial):

    params = {
        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                100,
                500
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                3,
                20
            ),

        "min_samples_split":
            trial.suggest_int(
                "min_samples_split",
                2,
                20
            ),

        "min_samples_leaf":
            trial.suggest_int(
                "min_samples_leaf",
                1,
                10
            ),

        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestClassifier(
        **params
    )

    tscv = TimeSeriesSplit(
        n_splits=5
    )

    scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        probs = model.predict_proba(
            X_val
        )[:,1]

        score = roc_auc_score(
            y_val,
            probs
        )

        scores.append(score)

    return sum(scores)/len(scores)


study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective_rf,
    n_trials=50
)

print(study.best_params)

[I 2026-06-12 14:13:49,964] A new study created in memory with name: no-name-4676b146-862f-4e16-9e27-5c3d9e1f5122
[I 2026-06-12 14:13:50,778] Trial 0 finished with value: 0.5144780532305195 and parameters: {'n_estimators': 115, 'max_depth': 18, 'min_samples_split': 13, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.5144780532305195.
[I 2026-06-12 14:13:52,825] Trial 1 finished with value: 0.5151330428536628 and parameters: {'n_estimators': 344, 'max_depth': 11, 'min_samples_split': 15, 'min_samples_leaf': 1}. Best is trial 1 with value: 0.5151330428536628.
[I 2026-06-12 14:13:55,390] Trial 2 finished with value: 0.5221740594807347 and parameters: {'n_estimators': 401, 'max_depth': 18, 'min_samples_split': 2, 'min_samples_leaf': 10}. Best is trial 2 with value: 0.5221740594807347.
[I 2026-06-12 14:13:57,185] Trial 3 finished with value: 0.5174676956340641 and parameters: {'n_estimators': 306, 'max_depth': 7, 'min_samples_split': 12, 'min_samples_leaf': 8}. Best is trial 2 with va

{'n_estimators': 401, 'max_depth': 6, 'min_samples_split': 14, 'min_samples_leaf': 7}


In [9]:
print("RF ROC-AUC:", study.best_value)
import joblib

joblib.dump(
    study.best_params,
    "models/rf_best_params.pkl"
)

RF ROC-AUC: 0.5236527832179876


['models/rf_best_params.pkl']

In [51]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(
    n_splits=7
)

In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np


def objective_xgb(trial):

    params = {

        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                100,
                600
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                3,
                10
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.3,
                log=True
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

        "min_child_weight":
            trial.suggest_int(
                "min_child_weight",
                1,
                10
            ),

        "gamma":
            trial.suggest_float(
                "gamma",
                0,
                5
            ),

        "random_state": 42,
        "eval_metric": "logloss",
        "n_jobs": -1
    }

    model = XGBClassifier(
        **params
    )

    scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(
            X_tr,
            y_tr
        )

        probs = model.predict_proba(
            X_val
        )[:, 1]

        score = roc_auc_score(
            y_val,
            probs
        )

        scores.append(score)

    return np.mean(scores)


xgb_study = optuna.create_study(
    direction="maximize"
)

xgb_study.optimize(
    objective_xgb,
    n_trials=50
)

print("Best ROC-AUC:")
print(xgb_study.best_value)

print("\nBest Params:")
print(xgb_study.best_params)

[I 2026-06-12 14:23:34,213] A new study created in memory with name: no-name-fef78569-44b3-4545-9b25-8cf26eb1d4e3
[I 2026-06-12 14:23:35,472] Trial 0 finished with value: 0.5241998514696928 and parameters: {'n_estimators': 493, 'max_depth': 5, 'learning_rate': 0.13325617135119577, 'subsample': 0.8875048030499693, 'colsample_bytree': 0.9176631073055592, 'min_child_weight': 3, 'gamma': 3.913285576464416}. Best is trial 0 with value: 0.5241998514696928.
[I 2026-06-12 14:23:36,886] Trial 1 finished with value: 0.5236093691183167 and parameters: {'n_estimators': 463, 'max_depth': 9, 'learning_rate': 0.014906086039215057, 'subsample': 0.730190395759805, 'colsample_bytree': 0.6649341767754766, 'min_child_weight': 8, 'gamma': 3.241981014401029}. Best is trial 0 with value: 0.5241998514696928.
[I 2026-06-12 14:23:37,994] Trial 2 finished with value: 0.5234696520266439 and parameters: {'n_estimators': 444, 'max_depth': 8, 'learning_rate': 0.015074778227103949, 'subsample': 0.9985143733656798, 'c

Best ROC-AUC:
0.5309877463059285

Best Params:
{'n_estimators': 367, 'max_depth': 3, 'learning_rate': 0.0127708291939357, 'subsample': 0.6315065333651801, 'colsample_bytree': 0.9361434838341645, 'min_child_weight': 2, 'gamma': 3.780633102534888}


In [13]:
import joblib

joblib.dump(
    xgb_study.best_params,
    "models/xgb_best_params.pkl"
)

['models/xgb_best_params.pkl']

In [17]:
import numpy as np
from sklearn.svm import SVC

def objective_svm(trial):

    params = {

        "C":
            trial.suggest_float(
                "C",
                0.01,
                100,
                log=True
            ),

        "gamma":
            trial.suggest_float(
                "gamma",
                0.0001,
                1,
                log=True
            ),

        "kernel":"rbf",

        "probability":True,

        "random_state":42
    }

    model = SVC(**params)

    scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train[train_idx]
        X_val = X_train[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        probs = model.predict_proba(X_val)[:,1]

        scores.append(
            roc_auc_score(
                y_val,
                probs
            )
        )

    return np.mean(scores)

svm_study = optuna.create_study(
    direction="maximize"
)

svm_study.optimize(
    objective_svm,
    n_trials=30
)

[I 2026-06-12 14:33:20,827] A new study created in memory with name: no-name-c2c722c7-5f89-4c56-89d6-7d4e2b870535
[I 2026-06-12 14:33:27,258] Trial 0 finished with value: 0.4937206496465773 and parameters: {'C': 0.017700283032955642, 'gamma': 0.049699860263491366}. Best is trial 0 with value: 0.4937206496465773.
[I 2026-06-12 14:33:33,578] Trial 1 finished with value: 0.5002000123916732 and parameters: {'C': 0.7300743971127254, 'gamma': 0.0026232203585717325}. Best is trial 1 with value: 0.5002000123916732.
[I 2026-06-12 14:33:39,774] Trial 2 finished with value: 0.5000654326607531 and parameters: {'C': 0.7034824522560368, 'gamma': 0.00297131080305522}. Best is trial 1 with value: 0.5002000123916732.
[I 2026-06-12 14:33:45,659] Trial 3 finished with value: 0.49590984301074537 and parameters: {'C': 0.01914618008956405, 'gamma': 0.00025845440092692613}. Best is trial 1 with value: 0.5002000123916732.
[I 2026-06-12 14:33:51,826] Trial 4 finished with value: 0.48573723927451484 and paramet

In [19]:
print("Best ROC-AUC:")
print(svm_study.best_value)

print("\nBest Params:")
print(svm_study.best_params)

Best ROC-AUC:
0.5233776304910684

Best Params:
{'C': 0.08901040169373603, 'gamma': 0.5355068984458151}


In [20]:
import joblib

joblib.dump(
    svm_study.best_params,
    "models/svm_best_params.pkl"
)

['models/svm_best_params.pkl']

In [39]:

from src.data_loader import load_data

(
    X_train,
    X_test,
    y_train,
    y_test,
    FEATURE_COLS
) = load_data()


import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import cross_val_score, TimeSeriesSplit

def objective(trial):

    # ── RF ────────────────────────────────────────────────
    rf_params = {
        "n_estimators"     : trial.suggest_int("rf_n_estimators", 100, 400),
        "max_depth"        : trial.suggest_int("rf_max_depth", 4, 10),
        "min_samples_split": trial.suggest_int("rf_min_samples_split", 10, 40),
        "min_samples_leaf" : trial.suggest_int("rf_min_samples_leaf", 5, 20),
    }

    # ── XGB ───────────────────────────────────────────────
    xgb_params = {
        "n_estimators"    : trial.suggest_int("xgb_n_estimators", 100, 400),
        "max_depth"       : trial.suggest_int("xgb_max_depth", 3, 6),
        "learning_rate"   : trial.suggest_float("xgb_lr", 0.01, 0.1, log=True),
        "subsample"       : trial.suggest_float("xgb_subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("xgb_colsample", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("xgb_min_child_weight", 3, 10),
        "gamma"           : trial.suggest_float("xgb_gamma", 0.0, 0.5),  # ← capped at 0.5
    }

    # ── SVM ───────────────────────────────────────────────
    svm_params = {
        "C"    : trial.suggest_float("svm_C", 0.1, 10.0, log=True),  # ← min 0.1
        "gamma": trial.suggest_float("svm_gamma", 0.001, 0.1, log=True),  # ← capped at 0.1
    }

    rf_model  = RandomForestClassifier(**rf_params, random_state=42, n_jobs=-1)
    xgb_model = XGBClassifier(**xgb_params, eval_metric="logloss",
                               random_state=42, n_jobs=-1)
    svm_model = SVC(**svm_params, kernel="rbf", probability=True, random_state=42)

    meta = LogisticRegression(C=0.1, max_iter=1000, random_state=42)

    stack = StackingClassifier(
        estimators=[("rf", rf_model), ("xgb", xgb_model), ("svm", svm_model)],
        final_estimator=meta,
        cv=5,
        stack_method="predict_proba",
        n_jobs=1
    )

    scores = cross_val_score(
        stack, X_train, y_train,
        cv=tscv, scoring="roc_auc", n_jobs=-1
    )
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1, show_progress_bar=True)
print(f"Trial 0 ROC-AUC : {study.best_value*100:.2f}%")

print(f"Best ROC-AUC : {study.best_value*100:.2f}%")
print(f"Best params  : {study.best_params}")


Total Features: 39
Train Rows : 5245
Test Rows  : 2485
Scaler saved


  0%|          | 0/1 [00:00<?, ?it/s]

Trial 0 ROC-AUC : 50.48%
Best ROC-AUC : 50.48%
Best params  : {'rf_n_estimators': 179, 'rf_max_depth': 5, 'rf_min_samples_split': 28, 'rf_min_samples_leaf': 15, 'xgb_n_estimators': 353, 'xgb_max_depth': 5, 'xgb_lr': 0.015548343228401281, 'xgb_subsample': 0.678455119122249, 'xgb_colsample': 0.7604720659069385, 'xgb_min_child_weight': 3, 'xgb_gamma': 0.15952883125839606, 'svm_C': 0.5391335097569013, 'svm_gamma': 0.014582269867961316}


In [41]:
# reload and re-encode df to match what load_data does
df_encoded = pd.get_dummies(df, columns=["ticker"], dtype=int)

print(f"Data range : {df_encoded['Date'].min().date()} → {df_encoded['Date'].max().date()}")
print(f"Total rows : {len(df_encoded)}")

print(f"\nFeature correlations with target (top 10):")
corr = df_encoded[FEATURE_COLS + ["target"]].corr()["target"].drop("target")
print(corr.abs().sort_values(ascending=False).head(10))

print(f"\nBottom 5 (weakest signal):")
print(corr.abs().sort_values(ascending=False).tail(5))

Data range : 2020-03-12 → 2026-06-09
Total rows : 7730

Feature correlations with target (top 10):
bb_upper          0.035081
ema_20            0.034636
Close             0.034611
ema_50            0.034585
Low               0.034513
Open              0.034458
High              0.034352
bb_lower          0.034218
atr               0.028438
ticker_SBIN.NS    0.022131
Name: target, dtype: float64

Bottom 5 (weakest signal):
macd                   0.001185
cdl_hammer             0.001125
macd_signal            0.000296
cdl_marubozu           0.000208
cdl_3white_soldiers         NaN
Name: target, dtype: float64


In [43]:
# check what target actually looks like
print(df[["Close", "tomorrow_close", "target"]].head(20))
print(f"\nTarget value counts: {df['target'].value_counts().to_dict()}")

# check if tomorrow_close is correctly shifted
print(f"\nSample close vs tomorrow_close:")
print(df[["Date", "ticker", "Close", "tomorrow_close", "target"]].head(10).to_string())

         Close  tomorrow_close  target
0   473.347778      492.183685       1
1   492.183685      452.285370       0
2   452.285370      448.856598       0
3   448.856598      431.267487       0
4   431.267487      408.646545       0
5   408.646545      453.287262       1
6   453.287262      393.662384       0
7   393.662384      420.090637       1
8   420.090637      481.919647       1
9   481.919647      474.772675       0
10  474.772675      474.505524       0
11  474.505524      458.853455       0
12  458.853455      495.946503       1
13  495.946503      481.118164       0
14  481.118164      479.782257       0
15  479.782257      537.069397       1
16  537.069397      530.857605       0
17  530.857605      543.236755       1
18  543.236755      529.521667       0
19  529.521667      512.021545       0

Target value counts: {1: 3979, 0: 3751}

Sample close vs tomorrow_close:
        Date       ticker       Close  tomorrow_close  target
0 2020-03-12  RELIANCE.NS  473.347778      49

In [45]:
# test correlation with different forward windows
for days in [1, 2, 3, 5, 10]:
    future = df.groupby("ticker")["Close"].shift(-days)
    target_nd = (future > df["Close"]).astype(int)
    
    corr = df[["rsi", "macd_hist", "bb_width", 
               "return_1d", "return_5d", "atr"]].corrwith(target_nd).abs()
    
    print(f"\n{days}-day forward target — avg feature correlation:")
    print(f"  mean : {corr.mean():.4f}")
    print(f"  max  : {corr.max():.4f} ({corr.idxmax()})")


1-day forward target — avg feature correlation:
  mean : 0.0096
  max  : 0.0287 (atr)

2-day forward target — avg feature correlation:
  mean : 0.0118
  max  : 0.0445 (atr)

3-day forward target — avg feature correlation:
  mean : 0.0131
  max  : 0.0434 (atr)

5-day forward target — avg feature correlation:
  mean : 0.0161
  max  : 0.0506 (atr)

10-day forward target — avg feature correlation:
  mean : 0.0224
  max  : 0.0589 (atr)


In [47]:
from src.data_loader import load_data

(
    X_train,
    X_test,
    y_train,
    y_test,
    FEATURE_COLS
) = load_data()
print(df.shape)

print(df["target"].value_counts(normalize=True))

print(df["Date"].min())
print(df["Date"].max())


Total Features: 46
Train Rows : 7430
Test Rows  : 2480
Scaler saved
(7730, 38)
target
1    0.514748
0    0.485252
Name: proportion, dtype: float64
2020-03-12 00:00:00
2026-06-09 00:00:00


In [48]:
import pandas as pd

df = pd.read_csv("all_stocks_features.csv")

df["Date"] = pd.to_datetime(df["Date"])

print(df.shape)
print("target" in df.columns)

(9910, 44)
True


In [50]:
from src.data_loader import load_data

(
    X_train,
    X_test,
    y_train,
    y_test,
    FEATURE_COLS
) = load_data()

print(df.shape)

print(df["target"].value_counts(normalize=True))

print(df["Date"].min())
print(df["Date"].max())


Total Features: 46
Train Rows : 7430
Test Rows  : 2480
Scaler saved
(9910, 44)
target
1    0.53996
0    0.46004
Name: proportion, dtype: float64
2018-05-25 00:00:00
2026-06-11 00:00:00


In [52]:
rf_study.optimize(
    objective_rf,
    n_trials=30
)

NameError: name 'rf_study' is not defined